In [2]:
import pandas as pd
import numpy as np

In [3]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="matfu21/yambda-50m-lag-features",
    repo_type="dataset",
    filename="listens.parquet",
)

listens = pd.read_parquet(path)
listens

c:\Users\Пр\Desktop\aaa\gp5\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,...,item_lag_like_cnt,item_lag_full_play_cnt,item_lag_skip_cnt,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio
0,100,6300205,6732,100,170,False,True,False,568.0,5.0,...,12.0,103.0,138.0,0.0,0.0,0.0,0.0,83.464789,47.592885,0.0
1,100,8508655,6732,55,170,False,False,False,1109.0,7.0,...,14.0,129.0,182.0,1.0,0.0,1.0,0.0,81.301172,46.525994,100.0
2,100,24750055,6732,100,170,False,True,False,2061.0,11.0,...,29.0,374.0,543.0,2.0,0.0,1.0,0.0,78.983018,46.107034,77.5
3,100,12127075,19712,97,280,False,True,False,1302.0,7.0,...,5.0,144.0,133.0,0.0,0.0,0.0,0.0,78.709677,56.202749,0.0
4,100,24502950,19712,24,280,False,False,True,1913.0,11.0,...,11.0,310.0,259.0,1.0,0.0,1.0,0.0,78.650810,59.609477,97.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22463550,1000000,19676790,9364985,100,200,False,True,False,156.0,3.0,...,43.0,1494.0,1083.0,1.0,0.0,1.0,0.0,88.358974,62.695924,100.0
22463551,1000000,22063255,9364985,100,200,False,True,False,308.0,3.0,...,52.0,1771.0,1294.0,2.0,0.0,2.0,0.0,91.746753,62.453086,100.0
22463552,1000000,19416280,9389535,100,205,False,True,False,94.0,3.0,...,37.0,696.0,467.0,0.0,0.0,0.0,0.0,92.914894,64.560065,0.0
22463553,1000000,20248090,9389535,100,205,False,True,False,164.0,3.0,...,39.0,716.0,482.0,1.0,0.0,1.0,0.0,88.926829,64.467666,100.0


In [4]:
def extract_preference_pairs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["row_id"] = np.arange(len(out))
    out = out.sort_values(["uid", "timestamp", "row_id"]).reset_index(drop=True)

    group = out.groupby("uid", sort=False)

    prev_like = group["is_like"].shift(1)
    next_like = group["is_like"].shift(-1)

    prev_full = group["is_full_play"].shift(1)
    next_full = group["is_full_play"].shift(-1)

    diff_prev = (
        prev_like.notna()
        & (
            out["is_like"].ne(prev_like)
            | out["is_full_play"].ne(prev_full)
        )
    )

    diff_next = (
        next_like.notna()
        & (
            out["is_like"].ne(next_like)
            | out["is_full_play"].ne(next_full)
        )
    )

    return (
        out[diff_prev | diff_next]
        .drop(columns=["row_id"])
        .reset_index(drop=True)
    )

listens = extract_preference_pairs(listens)


In [5]:
from datasets import DatasetDict, load_dataset


def load_precomputed_listens() -> pd.DataFrame:
    path = hf_hub_download(
        repo_id="matfu21/yambda-50m-lag-features",
        repo_type="dataset",
        filename="listens.parquet",
    )
    return pd.read_parquet(path)


def load_yambda_table(filename: str) -> pd.DataFrame:
    data = load_dataset(
        "yandex/yambda",
        data_dir="",
        data_files=f"{filename}.parquet",
    )

    assert isinstance(data, DatasetDict)
    return data["train"].to_pandas()

listens = load_precomputed_listens()

albums = load_yambda_table("album_item_mapping")
artists = load_yambda_table("artist_item_mapping")


In [6]:
import gc


def build_list_mapping(
    df: pd.DataFrame,
    key_col: str,
    value_col: str,
) -> dict:
    return (
        df
        .drop_duplicates([key_col, value_col])
        .sort_values([key_col, value_col])
        .groupby(key_col)[value_col]
        .agg(list)
        .to_dict()
    )


def join_item_artist_album(
    listens: pd.DataFrame,
    artists: pd.DataFrame,
    albums: pd.DataFrame,
) -> pd.DataFrame:
    out = listens.copy()

    artist_map = build_list_mapping(artists, "item_id", "artist_id")
    album_map = build_list_mapping(albums, "item_id", "album_id")

    out["artist_ids"] = out["item_id"].map(artist_map)
    out["album_ids"] = out["item_id"].map(album_map)

    out["artist_ids"] = [x if isinstance(x, list) else [] for x in out["artist_ids"]]
    out["album_ids"] = [x if isinstance(x, list) else [] for x in out["album_ids"]]

    return out

listens = join_item_artist_album(listens, artists, albums)

del artists, albums
gc.collect()

0

In [7]:
def temporal_train_test_split(
    df: pd.DataFrame,
    test_last_seconds: float,
    time_column: str = "timestamp",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    max_time = df[time_column].max()
    split_time = max_time - test_last_seconds

    train = df[df[time_column] < split_time].copy()
    test = df[df[time_column] >= split_time].copy()

    return train.reset_index(drop=True), test.reset_index(drop=True)

train_listens, test_listens = temporal_train_test_split(listens, test_last_seconds=30 * 24 * 60 * 60)

del listens
gc.collect()

0

In [8]:
import wandb


In [9]:
def _pairwise_accuracy(df, score_col, label_col):
    df = df.sort_values(["uid", "timestamp", "item_id"]).reset_index(drop=True)

    same_user = df["uid"].values[1:] == df["uid"].values[:-1]

    y_prev = df[label_col].values[:-1]
    y_next = df[label_col].values[1:]

    s_prev = df[score_col].values[:-1]
    s_next = df[score_col].values[1:]

    diff_mask = same_user & (y_prev != y_next)

    if diff_mask.sum() == 0:
        return 0.0

    y_diff = y_next[diff_mask] - y_prev[diff_mask]
    s_diff = s_next[diff_mask] - s_prev[diff_mask]

    return float((y_diff * s_diff > 0).mean())


In [10]:
test_listens.head(5)

,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,...,item_lag_skip_cnt,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio,artist_ids,album_ids
0,100,24750055,6732,100,170,False,True,False,2061.0,11.0,...,543.0,2.0,0.0,1.0,0.0,78.983018,46.107034,77.50,[205876],"[762872, 1947879]"
1,100,24502950,19712,24,280,False,False,True,1913.0,11.0,...,259.0,1.0,0.0,1.0,0.0,78.650810,59.609477,97.00,[452447],[2558848]
2,100,24688620,27230,100,140,False,True,False,2003.0,11.0,...,184.0,5.0,0.0,4.0,0.0,78.793310,57.365566,94.00,[700950],[1105922]
3,100,25251875,88111,35,255,False,False,True,2087.0,11.0,...,169.0,4.0,0.0,3.0,1.0,79.169142,51.079755,85.25,[480943],[1463938]
4,100,25279150,88111,38,255,False,False,True,2199.0,11.0,...,171.0,5.0,0.0,3.0,2.0,79.610732,51.027356,75.20,[480943],[1463938]


для того, чтобы потом смотреть, насколько наша модель предсказывает хорошо в сравнении, для начала построим бейзлайны

первый бейзлайн - это рандом. в целом рандом обычно предсказывает примерно 0.5 accuracy. построим и проверим это

In [11]:
test_random = test_listens[['uid', 'item_id', 'timestamp', 'is_like', 'is_full_play']].copy()
test_random['is_like'] = test_listens['is_like'].copy().astype(int)
test_random['is_full_play'] = test_listens['is_full_play'].copy().astype(int)

np.random.seed(17)
test_random['score'] = np.random.random(len(test_random))

test_random


,uid,item_id,timestamp,is_like,is_full_play,score
0,100,6732,24750055,0,1,0.294665
1,100,19712,24502950,0,0,0.530587
2,100,27230,24688620,0,1,0.191521
3,100,88111,25251875,0,0,0.067900
4,100,88111,25279150,0,0,0.786985
...,...,...,...,...,...,...
3054555,1000000,7479030,25277150,0,1,0.150814
3054556,1000000,7582497,25961030,0,1,0.370696
3054557,1000000,7696733,25276990,0,0,0.760995
3054558,1000000,8120372,25961615,0,1,0.750071


In [12]:
_pairwise_accuracy(test_random, 'score', 'is_like')

0.5055076991222651

In [13]:
_pairwise_accuracy(test_random, 'score', 'is_full_play')

0.49945597698708133

как мы и предполагали, значения pairwise_accuracy близки к 0.5

в целом, модель, которая должна быть лучше, чем рандом, это рекомендации популярных треков. потому что скорее всего, если песня популярна, то она нравится уже многим людям, значит многим и понравится. для начала посчитаем с популярностью как часто прослушиваемой песней

In [14]:
popular = train_listens.groupby('item_id').count().reset_index()[['item_id', 'uid']]
popular = popular.rename(columns={'uid': 'score'})
popular

,item_id,score
0,50,23
1,142,1
2,175,17
3,177,2
4,195,4
...,...,...
450354,9390522,1
450355,9390528,1
450356,9390541,19
450357,9390585,4


In [15]:
test_popular = test_listens[['uid', 'item_id', 'timestamp', 'is_like', 'is_full_play']].copy()
test_popular['is_like'] = test_listens['is_like'].copy().astype(int)
test_popular['is_full_play'] = test_listens['is_full_play'].copy().astype(int)

test_popular = test_popular.merge(popular, on='item_id', how='left')

test_popular['score'] = test_popular['score'].fillna(0)
test_popular

,uid,item_id,timestamp,is_like,is_full_play,score
0,100,6732,24750055,0,1,900.0
1,100,19712,24502950,0,0,582.0
2,100,27230,24688620,0,1,383.0
3,100,88111,25251875,0,0,293.0
4,100,88111,25279150,0,0,293.0
...,...,...,...,...,...,...
3054555,1000000,7479030,25277150,0,1,480.0
3054556,1000000,7582497,25961030,0,1,2350.0
3054557,1000000,7696733,25276990,0,0,1991.0
3054558,1000000,8120372,25961615,0,1,2219.0


In [16]:
_pairwise_accuracy(test_popular, 'score', 'is_like')

0.459184743964844

In [17]:
_pairwise_accuracy(test_popular, 'score', 'is_full_play')

0.5039553453987204

получилось +- так же, но попробуем теперь определять популярность как количество лайков, а не прослушиваний

In [18]:
popular_likes = train_listens.groupby('item_id')['is_like'].mean().reset_index()
popular_likes = popular_likes.rename(columns={'is_like': 'score'})
popular_likes

,item_id,score
0,50,0.0
1,142,0.0
2,175,0.0
3,177,0.0
4,195,0.0
...,...,...
450354,9390522,1.0
450355,9390528,0.0
450356,9390541,0.0
450357,9390585,0.0


In [19]:
test_popular_likes = test_listens[['uid', 'item_id', 'timestamp', 'is_like', 'is_full_play']].copy()
test_popular_likes['is_like'] = test_listens['is_like'].copy().astype(int)
test_popular_likes['is_full_play'] = test_listens['is_full_play'].copy().astype(int)

test_popular_likes = test_popular_likes.merge(popular_likes, on='item_id', how='left')

test_popular_likes['score'] = test_popular_likes['score'].fillna(0)
test_popular_likes

,uid,item_id,timestamp,is_like,is_full_play,score
0,100,6732,24750055,0,1,0.032222
1,100,19712,24502950,0,0,0.018900
2,100,27230,24688620,0,1,0.013055
3,100,88111,25251875,0,0,0.034130
4,100,88111,25279150,0,0,0.034130
...,...,...,...,...,...,...
3054555,1000000,7479030,25277150,0,1,0.002083
3054556,1000000,7582497,25961030,0,1,0.013191
3054557,1000000,7696733,25276990,0,0,0.006529
3054558,1000000,8120372,25961615,0,1,0.017575


In [20]:
_pairwise_accuracy(test_popular_likes, 'score', 'is_like')

0.4539393162293534

In [25]:
_pairwise_accuracy(test_popular_likes, 'score', 'is_full_play')

0.43787518025420036

в целом популярность показала себя чуть хуже. можно сказать, что популярность, посчитанная таким способом, не влияет на предпочтения пользователей (тк оно почти как рандом)

далее попробуем более сложную модель. попробуем сделать catboost. для неё нужно немного поправить признаки, так как она работает с категориальными.

In [22]:
DENSE_COLUMNS = [
    'user_lag_listen_cnt',
    'user_lag_like_cnt',
    'user_lag_full_play_cnt',
    'user_lag_skip_cnt',
    'item_lag_listen_cnt',
    'item_lag_like_cnt',
    'item_lag_full_play_cnt',
    'item_lag_skip_cnt',
    'ui_lag_listen_cnt',
    'ui_lag_like_cnt',
    'ui_lag_full_play_cnt',
    'ui_lag_skip_cnt',
    'user_lag_avg_played_ratio',
    'item_lag_avg_played_ratio',
    'ui_lag_avg_played_ratio',
]
SPARSE_COLUMNS = ['uid', 'item_id']

In [23]:
cols = DENSE_COLUMNS + SPARSE_COLUMNS
cols

['user_lag_listen_cnt',
 'user_lag_like_cnt',
 'user_lag_full_play_cnt',
 'user_lag_skip_cnt',
 'item_lag_listen_cnt',
 'item_lag_like_cnt',
 'item_lag_full_play_cnt',
 'item_lag_skip_cnt',
 'ui_lag_listen_cnt',
 'ui_lag_like_cnt',
 'ui_lag_full_play_cnt',
 'ui_lag_skip_cnt',
 'user_lag_avg_played_ratio',
 'item_lag_avg_played_ratio',
 'ui_lag_avg_played_ratio',
 'uid',
 'item_id']

In [28]:
train_sample = train_listens.sample(500000, random_state=17)

In [29]:
max_time = train_sample['timestamp'].max()
val_split = max_time - 7 * 24 * 60 * 60

X_val = train_sample[train_sample["timestamp"] >= val_split][cols]
y_val = train_sample[train_sample["timestamp"] >= val_split]["is_full_play"].astype(int)

X_train = train_sample[train_sample["timestamp"] < val_split][cols]
y_train = train_sample[train_sample["timestamp"] < val_split]["is_full_play"].astype(int)

In [24]:
max_time = train_listens['timestamp'].max()
val_split = max_time - 7 * 24 * 60 * 60

X_val = train_listens[train_listens["timestamp"] >= val_split][cols]
y_val = train_listens[train_listens["timestamp"] >= val_split]["is_full_play"].astype(int)

X_train = train_listens[train_listens["timestamp"] < val_split][cols]
y_train = train_listens[train_listens["timestamp"] < val_split]["is_full_play"].astype(int)

In [26]:
from catboost import CatBoostClassifier

In [33]:
model_catboost = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=4,
    early_stopping_rounds=100,
    l2_leaf_reg=5, 
    auto_class_weights="SqrtBalanced", 
    eval_metric="AUC",
    verbose=100,
    random_seed=17,
)
model_catboost.fit(
    X_train, y_train,
    cat_features=list(SPARSE_COLUMNS),
    eval_set=(X_val, y_val),
    use_best_model=True,
)

0:	test: 0.8126410	best: 0.8126410 (0)	total: 218ms	remaining: 7m 16s
100:	test: 0.8291812	best: 0.8291812 (100)	total: 22.9s	remaining: 7m 10s
200:	test: 0.8311023	best: 0.8311023 (200)	total: 54.3s	remaining: 8m 5s
300:	test: 0.8324189	best: 0.8324189 (300)	total: 1m 14s	remaining: 6m 57s
400:	test: 0.8332969	best: 0.8332969 (400)	total: 1m 36s	remaining: 6m 22s
500:	test: 0.8340397	best: 0.8340423 (499)	total: 1m 57s	remaining: 5m 51s
600:	test: 0.8345152	best: 0.8345175 (599)	total: 2m 23s	remaining: 5m 33s
700:	test: 0.8348808	best: 0.8348942 (690)	total: 2m 40s	remaining: 4m 58s
800:	test: 0.8352538	best: 0.8352546 (799)	total: 2m 57s	remaining: 4m 26s
900:	test: 0.8355690	best: 0.8355690 (900)	total: 3m 19s	remaining: 4m 3s
1000:	test: 0.8356833	best: 0.8357063 (989)	total: 3m 38s	remaining: 3m 37s
1100:	test: 0.8358311	best: 0.8358344 (1097)	total: 3m 54s	remaining: 3m 11s
1200:	test: 0.8359923	best: 0.8360063 (1198)	total: 4m 11s	remaining: 2m 47s
1300:	test: 0.8360863	best: 0

CatBoostClassifier(auto_class_weights='SqrtBalanced', depth=4, early_stopping_rounds=100, eval_metric='AUC', iterations=2000, l2_leaf_reg=5, learning_rate=0.05, random_seed=17, verbose=100)

In [34]:
model_catboost.get_best_iteration()

1266

In [37]:
X_test = test_listens[cols]

In [38]:
catboost_scores = model_catboost.predict_proba(X_test)[:, 1]
catboost_scores

array([0.52714353, 0.59456395, 0.65278093, ..., 0.83574822, 0.84168211,
       0.88607556], shape=(3054560,))

In [40]:
test_catboost = test_listens[['uid', 'item_id', 'timestamp', 'is_like', 'is_full_play']].copy()
test_catboost['is_like'] = test_catboost['is_like'].astype(int)
test_catboost['is_full_play'] = test_catboost['is_full_play'].astype(int)

test_catboost['score'] = catboost_scores
test_catboost

,uid,item_id,timestamp,is_like,is_full_play,score
0,100,6732,24750055,0,1,0.527144
1,100,19712,24502950,0,0,0.594564
2,100,27230,24688620,0,1,0.652781
3,100,88111,25251875,0,0,0.588680
4,100,88111,25279150,0,0,0.573378
...,...,...,...,...,...,...
3054555,1000000,7479030,25277150,0,1,0.901301
3054556,1000000,7582497,25961030,0,1,0.791121
3054557,1000000,7696733,25276990,0,0,0.835748
3054558,1000000,8120372,25961615,0,1,0.841682


In [41]:
_pairwise_accuracy(test_catboost, 'score', 'is_like')

0.47670447260138243

In [42]:
_pairwise_accuracy(test_catboost, 'score', 'is_full_play')

0.5645636078413837